In [1]:
import pandas as pd
import sqlite3
import io
from datetime import datetime

# ==============================================================================
# 1. CARGA Y GENERACIÓN DE DATASETS ORIGINALES (0.5 pts)
# ==============================================================================

# Datos proporcionados en el examen
clientes_raw = """id_cliente,cliente_info,email,telefono,edad,fecha_nacimiento
101,Diego Zuluaga - Profesor,diego@correo.com,3001112233,42,1984-07-07
102,ana gomez - estudiante,ana@gmail.com,,21,2005-02-15
103,CARLOS RUIZ - PROFESOR,cruiz@empresa.co,3112223344,38,
104,Maria Lopez - Estudiante,,3154445566,20,2006-11-20
105,LUIS PEREZ - Externo,luis.p@correo.com,,45,1981-05-10"""

proveedores_raw = """nit_proveedor,empresa_ciudad,contacto,telefono,email
900111,CoopCafe - Bogota,Carlos Perez,3005551122,carlos@coop.com
800222,Insumos Panaderos - Medellin,,3114445566,
700333,Lacteos El Prado - Cali,Ana Rojas,,ana@prado.com
600444,Distribuidora Sabana - Chia,Luis Gomez,3208889900,luis@sabana.com
500555,Salsas y Especias - Bogota,,3157778899,"""

productos_raw = """id_producto,producto_info,precio,stock,nit_proveedor
1,Cafe Tostado - Grano,25000,100,900111
2,Pan Blandito - Panaderia,3000,50,800222
3,Leche Entera - Lacteos,4500,80,700333
4,Miel Organica - Dulces,12000,20,500555
5,Aroma Canela - Especias,8000,40,500555"""

# Guardar archivos sucios originales
with open('clientes_sucios.csv', 'w') as f: f.write(clientes_raw)
with open('proveedores_sucios.csv', 'w') as f: f.write(proveedores_raw)
with open('productos_sucios.csv', 'w') as f: f.write(productos_raw)

# ==============================================================================
# 2. PANDAS Y NORMALIZACIÓN (1.5 pts)
# ==============================================================================

# --- Limpieza de Clientes ---
df_clientes = pd.read_csv(io.StringIO(clientes_raw))
# 1NF: Atomicidad en cliente_info
df_clientes[['nombre', 'tipo']] = df_clientes['cliente_info'].str.split(' - ', expand=True)
df_clientes['nombre'] = df_clientes['nombre'].str.title()
df_clientes['tipo'] = df_clientes['tipo'].str.capitalize()
# Manejo de nulos y tipos
df_clientes['fecha_nacimiento'] = df_clientes['fecha_nacimiento'].fillna('1900-01-01')
df_clientes['email'] = df_clientes['email'].fillna('sin_correo@u.edu.co')
df_clientes = df_clientes.drop(columns=['cliente_info'])

# --- Limpieza de Proveedores ---
df_prov = pd.read_csv(io.StringIO(proveedores_raw))
df_prov[['empresa', 'ciudad']] = df_prov['empresa_ciudad'].str.split(' - ', expand=True)
df_prov['contacto'] = df_prov['contacto'].fillna('Departamento Comercial')
df_prov = df_prov.drop(columns=['empresa_ciudad'])

# --- Limpieza de Productos ---
df_prod = pd.read_csv(io.StringIO(productos_raw))
df_prod[['nombre_producto', 'categoria']] = df_prod['producto_info'].str.split(' - ', expand=True)
df_prod = df_prod.drop(columns=['producto_info'])

# Exportar datasets limpios
df_clientes.to_csv('clientes_limpios.csv', index=False)
df_prov.to_csv('proveedores_limpios.csv', index=False)
df_prod.to_csv('productos_limpios.csv', index=False)

# ==============================================================================
# 3. MIGRACIÓN SQLITE (1.0 pts)
# ==============================================================================

db_name = 'parcial_2_cafeteria.db'
conn = sqlite3.connect(db_name)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;") # Activación de FK

# Carga de tablas desde DataFrames
df_clientes.to_sql('clientes', conn, if_exists='replace', index=False)
df_prov.to_sql('proveedores', conn, if_exists='replace', index=False)
df_prod.to_sql('productos', conn, if_exists='replace', index=False)

# Crear tabla Ventas (Arquitectura Relacional)
cursor.execute('''
CREATE TABLE IF NOT EXISTS ventas (
    id_venta INTEGER PRIMARY KEY AUTOINCREMENT,
    id_cliente INTEGER,
    id_producto INTEGER,
    cantidad INTEGER,
    total REAL,
    fecha TEXT,
    FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente),
    FOREIGN KEY (id_producto) REFERENCES productos(id_producto)
)''')
conn.commit()

# ==============================================================================
# 4. BONO: MODULARIZACIÓN CON POO (+1.0 pt)
# ==============================================================================

class GestionCafeteria:
    def __init__(self, db):
        self.db = db

    def registrar_venta(self, id_cli, id_prod, cant):
        try:
            with sqlite3.connect(self.db) as c:
                cursor = c.cursor()
                # Consultar precio del producto
                cursor.execute("SELECT precio FROM productos WHERE id_producto = ?", (id_prod,))
                precio = cursor.fetchone()[0]
                total = precio * cant
                fecha = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                
                cursor.execute("INSERT INTO ventas (id_cliente, id_producto, cantidad, total, fecha) VALUES (?,?,?,?,?)",
                               (id_cli, id_prod, cant, total, fecha))
                print(f"✅ [CREATE] Venta registrada exitosamente.")
        except Exception as e:
            print(f"❌ Error: {e}")

    def ver_reporte_ventas(self):
        with sqlite3.connect(self.db) as c:
            # JOIN entre Ventas, Clientes y Productos
            query = '''
            SELECT v.id_venta, cli.nombre, prod.nombre_producto, v.cantidad, v.total
            FROM ventas v
            JOIN clientes cli ON v.id_cliente = cli.id_cliente
            JOIN productos prod ON v.id_producto = prod.id_producto
            '''
            df_reporte = pd.read_sql(query, c)
            print("\n--- REPORTE DE INTELIGENCIA DE NEGOCIOS (JOIN) ---")
            print(df_reporte)

    def actualizar_stock(self, id_prod, nuevo_stock):
        with sqlite3.connect(self.db) as c:
            c.execute("UPDATE productos SET stock = ? WHERE id_producto = ?", (nuevo_stock, id_prod))
            print(f"✅ [UPDATE] Stock actualizado para producto {id_prod}.")

    def eliminar_venta(self, id_v):
        with sqlite3.connect(self.db) as c:
            c.execute("DELETE FROM ventas WHERE id_venta = ?", (id_v,))
            print(f"✅ [DELETE] Venta {id_v} eliminada.")

# --- PRUEBAS DEL SISTEMA ---
app = GestionCafeteria(db_name)
app.registrar_venta(101, 1, 5) # Venta para Diego
app.registrar_venta(102, 2, 10) # Venta para Ana
app.ver_reporte_ventas()
app.actualizar_stock(1, 95)
app.eliminar_venta(2)
conn.close()

✅ [CREATE] Venta registrada exitosamente.
✅ [CREATE] Venta registrada exitosamente.

--- REPORTE DE INTELIGENCIA DE NEGOCIOS (JOIN) ---
   id_venta         nombre nombre_producto  cantidad     total
0         1  Diego Zuluaga    Cafe Tostado         5  125000.0
1         2      Ana Gomez    Pan Blandito        10   30000.0
✅ [UPDATE] Stock actualizado para producto 1.
✅ [DELETE] Venta 2 eliminada.
